# Projeto SPA - Alocação de alunos em projetos
**Teoria e Aplicação de Grafos, Turma A, 2026/1 — Prof. Díbio**  
Este notebook implementa e demonstra o pipeline completo de alocação de alunos a projetos usando o problema de *Student-Project Allocation* (SPA), baseado em Abraham, Irving & Manlove (2007).

### Integrantes
- Daniel Florencio Hollenbach, matrícula 241020859
- Mateus Valério, matrícula 190035161
- Maylla Krislainy, matrícula 190043873.

### Estrutura
1. [Leitura e validação dos dados de entrada](#sec-leitura)
2. [Construção do grafo bipartido](#sec-grafo)
3. [Emparelhamento estável inicial](#sec-emparelhamento)
4. [Iterações de melhoria](#sec-iteracoes)
5. [Métricas finais](#sec-metricas)

## Configuração do ambiente

In [ ]:
%matplotlib inline
import sys
from functools import partial
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

# Injeta a raiz no Path dinamicamente
ROOT = Path.cwd()
if str(ROOT) not in sys.path:  # type: ignore
    sys.path.append(str(ROOT))  # type: ignore

import spa
from spa import (
    parse_input_file,
    build_bipartite_graph,
    run_gale_shapley,
    build_matching_state,
    run_iterations,
    compute_all_preference_indices,
    build_final_matching_matrix,
    plot_bipartite_iteration,
    plot_preference_index_summary,
    save_matching_matrix_csv,
    is_stable_matching,
)

print("[OK] Módulos importados com sucesso.")

<a id="sec-leitura"></a>
## Leitura e validação dos dados de entrada
O arquivo `entradaProj2.26TAG.txt` contém duas seções:
- **Projetos**: `(código, nº vagas, nota mínima exigida)` - 50 projetos
- **Alunos**: `(código, preferências ordenadas, nota agregada [3–5])` - 200 alunos

Cada aluno pode indicar até 3 projetos em ordem de preferência. A nota agregada representa a avaliação do histórico e disponibilidade do aluno: 3 = suficiente, 4 = muito boa, 5 = excelente.

> **Nota sobre o arquivo de entrada:** foram identificadas inconsistências nos dados fornecidos, projetos inexistentes referenciados nas preferências (P51, P52, P53, P54) e preferências duplicadas em alguns alunos. Essas ocorrências são logadas como avisos e tratadas sem interromper o pipeline.

In [ ]:
INPUT_FILE = "data/entradaProj2.26TAG.txt"

# Faz o parse do arquivo e instancia objetos Projeto e Aluno
projetos, alunos = parse_input_file(INPUT_FILE)

# Valida integridade dos dados: projetos inexistentes, notas inválidas,
# preferências duplicadas. Loga avisos mas não interrompe a execução.
spa.validate_parsed_data(projetos, alunos)
total_vagas = sum(p.num_vagas for p in projetos)

print(f"  • Projetos carregados:  {len(projetos)} projetos (Total de {total_vagas} vagas ofertadas)")
print(f"  • Alunos carregados:    {len(alunos)} candidatos inscritos")
print(f"  • Relação Candidato/Vaga: {(len(alunos)/total_vagas if total_vagas > 0 else 0):.2f}")

# Mini-Tabela de Inspeção — Projetos (Amostra Top 5)
df_projetos = pd.DataFrame([
    {
        "Código": p.cod,
        "Vagas Ofertadas": p.num_vagas,
        "Nota Mínima Exigida": p.nota_min
    }
    for p in projetos[:5]
])


<a id="sec-grafo"></a>
## Construção do grafo bipartido
O problema SPA é modelado como um grafo bipartido **G = (A ∪ P, E)**, onde:
- **A** = conjunto de alunos (partição esquerda, `bipartite=0`)
- **P** = conjunto de projetos (partição direita, `bipartite=1`)
- Uma aresta **(a, p) ∈ E** existe se o projeto `p` aparece na lista de preferências do aluno `a`

Cada aresta carrega o atributo `preferencia` ∈ {1, 2, 3}, indicando a posição na lista do aluno. A nota mínima **não filtra arestas** nesta etapa, é critério aplicado dinamicamente durante as propostas do Gale-Shapley e na busca por caminhos aumentantes.

Cada nó de projeto recebe também o atributo `candidatos`, lista de todos os alunos que o incluíram nas preferências, usado para calcular o ranking do aluno na matriz final.

In [ ]:
# Constrói G = (A ∪ P, E) com nós Aluno/Projeto como objetos hasheáveis.
# Arestas são criadas para cada preferência válida (projeto existente na lista).
graph = build_bipartite_graph(projetos, alunos)

total_nos = graph.number_of_nodes()
total_arestas = graph.number_of_edges()
media_pref = total_arestas / len(alunos) if len(alunos) > 0 else 0

print("=== TOPOLOGIA DO GRAFO BIPARTIDO DE ENTRADA ===")
print(f"  • Total de Vértices |V|: {total_nos} ({len(alunos)} alunos + {len(projetos)} projetos)")
print(f"  • Total de Arestas  |E|: {total_arestas}")
print(f"  • Grau Médio (Alunos):   {media_pref:.2f} preferências indicadas por aluno")


<a id="sec-emparelhamento"></a>
## Emparelhamento estável inicial
Executa a variação descrita abaixo e produz o matching estável **M₀**.
O grafo resultante é plotado com cores:
- **Azul** — arestas no matching M₀
- **Vermelho tracejado** — rejeições do GS
- **Cinza** — arestas não propostas

### Variação do Gale-Shapley utilizada
A implementação segue a variação **SPA-Students** descrita em Abraham, Irving & Manlove (2007), com uma adaptação por mérito:

- **Propostas partem dos alunos**, em ordem crescente de matrícula (execução determinística)
- Cada aluno propõe sequencialmente aos projetos em sua lista de preferência
- O projeto **aceita** se o aluno atende à `nota_min` **e** há vaga disponível
- Se o projeto está **cheio** mas o candidato tem nota **estritamente maior** que o pior alocado, o pior é **expulso** e volta à fila de livres *(substituição por mérito)*
- O algoritmo termina quando não há mais alunos livres com propostas pendentes

O resultado M₀ é um matching estável: nenhum par (aluno, projeto) bloqueante existe, pois todo aluno não alocado foi rejeitado por todos os seus projetos preferidos (por nota insuficiente ou projeto cheio com candidatos melhores). A verificação foi feita através da execução da função **is_stable_matching**.

In [ ]:
# Executa GS: alunos propõem em ordem de matrícula, projetos aceitam/rejeitam
# por nota mínima e capacidade, com substituição por mérito quando cheio
matching, _, rejected_edges, allocated_projects, free_students = run_gale_shapley(projetos, alunos)
spa.is_stable_matching(matching, graph)

# MatchingState encapsula o estado completo: pares, rejeições e alunos livres
state = build_matching_state(
    matching=matching,
    rejected_edges=rejected_edges,
    allocated_projects=allocated_projects,
    free_students=free_students,
    iteration=0,
)
print("\n=== RESUMO DO EMPARELHAMENTO ESTÁVEL INICIAL (M₀) ===")
print(f"  • Alunos alocados:   {len(matching)} de {len(alunos)} candidatos")
print(f"  • Alunos livres:     {len(free_students)}")
print(f"  • Rejeições por vaga ou corte de nota: {len(rejected_edges)}")

df_m0 = pd.DataFrame([
    {
        "Aluno": a.cod, 
        "Nota Agregada": a.nota, 
        "Projeto Alocado": p.cod,
        # Pega a posição em que o projeto estava na lista do aluno (1º, 2º ou 3º)
        "Opção": f"{a.preferencia.index(p) + 1}ª" if p in a.preferencia else "-"
    }

    for a, p in matching[:10]
])

print("\n--- Amostra dos Primeiros Alocados em M₀ (Top 10) ---")
display(df_m0)
print(f"*(Exibindo 10 de {len(matching)} pares gerados na Fase 1)*\n")

# Gera e exibe o grafo da iteração 0 (estado pós-GS)
initial_log = {
    "matched_edges": [(s, p) for s, p in matching],
    "proposed_edges": [],
    "rejected_edges": rejected_edges,
}
plot_bipartite_iteration(state, initial_log, 0, graph)
display(Image("output/iteracao_0.png", width=700)) # Grafo salvo em output/iteracao_0.png

<a id="sec-iteracoes"></a>
## Iterações de melhoria
O matching M₀ produzido pelo GS é melhorado ao longo de **10 iterações** via busca por caminhos M-aumentantes (algoritmo de caminhos alternados).

**Lógica de cada iteração:**
1. Ordena alunos livres por matrícula (ordem fixa, determinística)
2. Para o primeiro aluno livre, busca um **caminho M-aumentante** via BFS:
   - Parte do aluno livre por uma aresta **fora** do matching (aluno → projeto)
   - Se o projeto está cheio, segue por uma aresta **dentro** do matching (projeto → aluno alocado)
   - Alterna até encontrar um projeto com **vaga disponível**
3. Se encontrado: **aumenta** o matching invertendo o status de cada aresta do caminho (|M| cresce em 1)
4. O grafo é plotado **antes** da augmentação para exibir proposta ativa, matching atual e rejeições
5. Atualiza lista de alunos livres para a próxima rodada

Se nenhum caminho aumentante existe, o matching é **máximo** e as iterações restantes não alteram o resultado.
> Os grafos de cada iteração são exibidos individualmente abaixo, após a execução.

É válido mencionar que, segundo a especificação, o algorítimo buscar por caminhos M-aumentantes mas não garante que o emparelhamento resultante é estável. Podemos comprovar isso novamente através da execução da função **is_stable_matching**.

In [ ]:
# partial fixa o argumento 'graph' no callback, pois run_iterations
# chama on_iteration_end(state, iteration_log, iteration_number).
plot_iteration = partial(plot_bipartite_iteration, graph=graph)

iteration_states = run_iterations(
    graph,
    state,
    all_alunos=alunos,          # necessário para atualizar free_students a cada rodada
    n_iterations=10,
    on_iteration_end=plot_iteration,  # plota antes de aplicar augmentação
)

final_state = iteration_states[-1] if iteration_states else state
# veriricação de emparelhamento estável
is_stable_matching(final_state.matching, graph)

print("=" * 60)
print(f"  • Matching final |M|: {len(final_state.matching)} alunos alocados")
print(f"  • Alunos sem projeto: {len(final_state.free_students)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

def plotar_iteracoes(iteracoes, pasta="output"):
    """
    Plota uma grade horizontal com as imagens das iterações informadas.
    Exemplo: plotar_iteracoes(range(1, 6)) ou plotar_iteracoes([1, 3, 10])
    """
    lista_iteracoes = list(iteracoes)
    num_colunas = len(lista_iteracoes)
    
    if num_colunas == 0:
        print("⚠️ Nenhuma iteração informada.")
        return

    largura_dinamica = num_colunas * 2.2
    
    fig, axs = plt.subplots(
        1, num_colunas, 
        figsize=(largura_dinamica, 11), 
        gridspec_kw={'wspace': 0.02}
    )

    pasta_path = Path(pasta)
    
    print("[INFO]: Carregando imagens...")
    for idx, i in enumerate(lista_iteracoes):
        ax = axs[idx] if num_colunas > 1 else axs
        arquivo = pasta_path / f"iteracao_{i}.png"
        
        if arquivo.exists():
            ax.imshow(mpimg.imread(arquivo))
            ax.set_title(f"Iteração {i}", fontweight="bold", fontsize=12, pad=10)
        else:
            ax.text(0.5, 0.5, f"Iteração {i}\nnão encontrada", ha='center', va='center', color='red')
        
        ax.axis('off')

    plt.show()


### Grafos das iterações 1–5

In [ ]:
plotar_iteracoes(range(1, 6))

### Grafos das iterações 6–10

In [ ]:
plotar_iteracoes(range(6, 11))

<a id="sec-metricas"></a>
## Métricas finais

### Índice de preferência por projeto
Para cada projeto `p`, calcula a média da posição de preferência dos alunos alocados:
- **1.0** = todos os alunos tinham `p` como 1ª escolha (ideal)
- **2.0** = média de 2ª escolha
- **3.0** = todos tinham `p` como última opção

Quanto **menor** o índice, maior a satisfação dos alunos alocados naquele projeto.

### Matriz de emparelhamento
Para cada aluno, registra:
- Projeto emparelhado
- Classificação do aluno na lista do projeto (ranking por nota decrescente)
- Classificação do projeto na lista de preferência do aluno (1ª, 2ª ou 3ª escolha)

In [ ]:
# Cálculo dos índices de preferência (média das escolhas por projeto)
preference_indices = compute_all_preference_indices(final_state, graph)

# Construção da matriz final com as colunas exigidas pelo enunciado
matching_matrix = build_final_matching_matrix(final_state, graph)

print("=== Índices de Preferência por Projeto (Amostra Top 15) ===")
# Ordena por código do projeto e filtra para mostrar um resumo visual limpo
projetos_ordenados = sorted(preference_indices.items(), key=lambda x: x[0].cod)
for projeto, indice in projetos_ordenados[:15]:
    print(f"  {projeto.cod}: {indice:.2f}° opção em média")
print(
    f"  ... (+ {len(projetos_ordenados) - 15} projetos omitidos no print para clareza)"
)

# Gráfico de barras do índice de preferência por projeto
plot_preference_index_summary(preference_indices)
display(Image("output/preference_index.png", width=1000))


print("=== Matriz de Emparelhamento Final (Primeiros 15 Alunos) ===")
display(matching_matrix.head(15))
save_matching_matrix_csv(matching_matrix)
print("\n[OK] CSV completo com os 200 alunos salvo em: output/matching_matrix.csv")

## Encerramento
O pipeline implementado resolve o problema de *Student-Project Allocation* em duas fases:

**Fase 1 — Gale-Shapley:** Produz um matching estável M₀ onde nenhum aluno prefere um projeto que o aceitaria e nenhum projeto teria motivo para trocar um alocado por um candidato rejeitado. A variação por mérito (substituição pelo aluno de maior nota) maximiza a qualidade dos alocados dentro das restrições de capacidade.

**Fase 2 — Caminhos aumentantes:** Tenta ampliar M₀ iterativamente, alocando alunos que ficaram sem projeto via caminhos alternados no grafo bipartido. O processo é determinístico e termina quando o matching é máximo (nenhum caminho aumentante existe).

**Limitação observada:** O matching final aloca 63 de 200 alunos (31,5%). Isso reflete as restrições dos dados de entrada — muitos projetos exigem `nota_min=5` enquanto a maioria dos alunos tem nota 3 ou 4, tornando o conjunto de pares compatíveis restrito. O algoritmo encontra o **matching máximo estável** dado o input fornecido.